In [1]:
from transformers import AutoModelForQuestionAnswering, BertTokenizer

model_id = 'google-bert/bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_id)
model = AutoModelForQuestionAnswering.from_pretrained(model_id)
model.eval()

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params 

BertForQuestionAnswering(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elem

Stanford Question Answering Dataset, https://huggingface.co/datasets/squad


In [3]:
from datasets import load_dataset

samples_count = 200
squad = load_dataset("rajpurkar/squad", split="validation[:" + str(samples_count) + "]")

In [4]:
# Data directory
import os

output_dir = os.path.join(".", "onnx_models")
onnx_model_filename = 'bert-base-uncased.onnx'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
export_model_path = os.path.join(output_dir, onnx_model_filename)

In [5]:
tokenized_inputs = tokenizer(squad["question"][0], squad["context"][0], return_tensors="pt")
inputs = {
        'input_ids':  tokenized_inputs['input_ids'],
        'input_mask': tokenized_inputs['attention_mask'],
        'segment_ids': tokenized_inputs['token_type_ids']
    }

In [7]:
import torch

with torch.no_grad():
    symbolic_names = {0: 'batch_size', 1: 'max_seq_len'}
    torch.onnx.export(
        model,
        args=tuple(inputs.values()),
        f=export_model_path,
        opset_version=15,
        do_constant_folding=True,
        input_names=[
            'input_ids',
            'input_mask',
            'segment_ids'
        ],
        output_names=['start', 'end'],
        dynamic_axes={
            'input_ids': symbolic_names,
            'input_mask' : symbolic_names,
            'segment_ids' : symbolic_names,
            'start' : symbolic_names,
            'end' : symbolic_names
        }
    )

/var/folders/pz/jgxccbqs40b3pxwj2v5g_gmh0000gn/T/ipykernel_65782/3593651785.py:5: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0612 14:35:56.490000 65782 env-llm/lib/python3.13/site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 15 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0612 14:35:57.229000 65782 env-llm/lib/python3.13/site-packages/torch/onnx/_internal/exporter/_registration.py:110] torchvision is not instal

[torch.onnx] Obtain model graph for `BertForQuestionAnswering([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `BertForQuestionAnswering([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/opt/homebrew/Cellar/python@3.13/3.13.2/Frameworks/Python.framework/Versions/3.13/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 15).
Failed to convert the model to the target version 15 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/Users/joewilkinson/Projects/scratchllm/env-llm/lib/python3.13/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/Users/joewilkinson/Projects/scratchllm/env-llm/lib/python3.13/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_a

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/Users/joewilkinson/Projects/scratchllm/env-llm/lib/python3.13/site-packages/torch/onnx/_internal/exporter/_onnx_program.py:487: UserWarning: # The axis name: max_seq_len will not be used, since it shares the same shape constraints with another axis: max_seq_len.
  rename_mapping = _dynamic_shapes.create_rename_mapping(
/Users/joewilkinson/Projects/scratchllm/env-llm/lib/python3.13/site-packages/torch/onnx/_internal/exporter/_onnx_program.py:487: UserWarning: # The axis name: batch_size will not be used, since it shares the same shape constraints with another axis: batch_size.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


In [8]:
# Check model
from onnx.checker import check_model

check_model(export_model_path, full_check=True)

To validate the full architecture, run inference on a few samples (or the full test set) with both the original model and the ONNX-converted one, and compare their outputs. NumPy provides the allclose function for this:


In [9]:
import onnxruntime as rt
import torch

session = rt.InferenceSession(export_model_path)

outputs = []
ort_outputs = []

for i in range(10):
    tokenized = tokenizer(squad["question"][i], squad["context"][i], return_tensors="pt")

    # PyTorch inference
    with torch.no_grad():
        pt_out = model(**tokenized)
    outputs.append(pt_out.start_logits)
    outputs.append(pt_out.end_logits)

    # ONNX Runtime inference
    ort_inputs = {
        "input_ids": tokenized["input_ids"].numpy(),
        "input_mask": tokenized["attention_mask"].numpy(),
        "segment_ids": tokenized["token_type_ids"].numpy(),
    }
    ort_out = session.run(None, ort_inputs)
    ort_outputs.append(ort_out[0])  # start logits
    ort_outputs.append(ort_out[1])  # end logits

In [10]:
import numpy as np

for i in range(10):
    np.allclose(ort_outputs[i], outputs[i].cpu(), rtol=1e-05, atol=1e-04)

In [11]:
import onnxruntime

sess_options = onnxruntime.SessionOptions()
session = onnxruntime.InferenceSession(
    export_model_path,
    sess_options,
    providers=['CPUExecutionProvider']
)

In [13]:
inputs = tokenizer(squad["question"][0], squad["context"][0], return_tensors="np")
ort_inputs = {
    'input_ids':  inputs['input_ids'],
    'input_mask': inputs['attention_mask'],
    'segment_ids': inputs['token_type_ids']
}
ort_outputs = session.run(None, ort_inputs)

In [16]:
import time
import numpy as np
import torch
import onnxruntime


def benchmark_pytorch(model, tokenizer, dataset, n_samples):
    times = []
    for i in range(n_samples):
        tokenized = tokenizer(dataset["question"][i], dataset["context"][i], return_tensors="pt")
        start = time.perf_counter()
        with torch.no_grad():
            model(**tokenized)
        times.append((time.perf_counter() - start) * 1000)
    avg = np.mean(times)
    print(f"PyTorch CPU Average inference time = {avg:.2f} ms")
    return avg


def benchmark_onnx(model_path, tokenizer, dataset, n_samples, label="OnnxRuntime CPU"):
    session = onnxruntime.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    times = []
    for i in range(n_samples):
        tokenized = tokenizer(dataset["question"][i], dataset["context"][i], return_tensors="np")
        ort_inputs = {
            "input_ids": tokenized["input_ids"],
            "input_mask": tokenized["attention_mask"],
            "segment_ids": tokenized["token_type_ids"],
        }
        start = time.perf_counter()
        session.run(None, ort_inputs)
        times.append((time.perf_counter() - start) * 1000)
    avg = np.mean(times)
    print(f"{label} Average inference time = {avg:.2f} ms")
    return avg

In [17]:
pt_avg = benchmark_pytorch(model, tokenizer, squad, samples_count)
onnx_avg = benchmark_onnx(export_model_path, tokenizer, squad, samples_count, label="OnnxRuntime CPU")
print(f"Speedup: {pt_avg / onnx_avg:.2f}x")

PyTorch CPU Average inference time = 87.07 ms
OnnxRuntime CPU Average inference time = 83.08 ms
Speedup: 1.05x


In [20]:
import onnxruntime

optimized_model_path = os.path.join(output_dir, 'bert-base-uncased.onnx_opt_cpu.onnx')

sess_options = onnxruntime.SessionOptions()
sess_options.optimized_model_filepath = optimized_model_path
sess_options.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL

# Running a session with these options triggers optimization and saves the result
onnxruntime.InferenceSession(export_model_path, sess_options, providers=["CPUExecutionProvider"])
print(f"Optimized model saved to {optimized_model_path}")

Optimized model saved to ./onnx_models/bert-base-uncased.onnx_opt_cpu.onnx


2026-06-12 14:58:16.523 Python[65782:25411991] 2026-06-12 14:58:16.522502 [W:onnxruntime:, inference_session.cc:2670 Initialize] Serializing optimized model with Graph Optimization level greater than ORT_ENABLE_EXTENDED and the NchwcTransformer enabled. The generated model may contain hardware specific optimizations, and should only be used in the same environment the model was optimized in.


In [21]:
opt_avg = benchmark_onnx(optimized_model_path, tokenizer, squad, samples_count, label="OnnxRuntime CPU (optimized)")
print(f"Speedup vs PyTorch: {pt_avg / opt_avg:.2f}x")
print(f"Speedup vs unoptimized ONNX: {onnx_avg / opt_avg:.2f}x")

OnnxRuntime CPU (optimized) Average inference time = 82.61 ms
Speedup vs PyTorch: 1.05x
Speedup vs unoptimized ONNX: 1.01x
